In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import defaultdict
from data_preprocess import encode_features, construct_df, PCA, FeatureSelection

In [ ]:
class NaiveBayesClassifier:
    def __init__(self, data=None):
        self.edidble_counts = 0
        self.poisonous_counts = 0
        self.edidble_prob = 0.5
        self.poisonous_prob = 0.5
        self.data = data
        self.edidble_features_likelihoods = {}
        self.poisonous_features_likelihoods = {}
        
    def loadData(self, data):
        self.data = data

    def compute_initial_probs(self):
        data = self.data
        nrecords = len(data)
        
        counts = data['class'].value_counts()
        edidble_counts = counts['e']
        poisonous_counts = counts['p']
        
        total = edidble_counts + poisonous_counts
        
        self.edidble_counts = edidble_counts
        self.poisonous_counts = poisonous_counts
        
        self.edidble_prob = edidble_counts / total
        self.poisonous_prob = poisonous_counts / total
    
        
        return self.edidble_prob, self.poisonous_prob
    
    def compute_histograms(self):
        df = self.data
        
        edidble_features_freq = defaultdict(dict)
        poisonous_features_freq = defaultdict(dict)
        total_edible_observations = self.edidble_counts * 22
        total_poisonous_observations = self.poisonous_counts * 22
        
        for _, row in df.iterrows():
            label = row['class']
            
            for feature in df.columns:
                if feature == 'class': continue
                
                value = row[feature]
                if label == 'e':
                    edidble_features_freq[feature][value] = edidble_features_freq[feature].get(value, 0) + 1
                else:
                    poisonous_features_freq[feature][value] = poisonous_features_freq[feature].get(value, 0) + 1
                    
        return edidble_features_freq, poisonous_features_freq, total_edible_observations, total_poisonous_observations
    

    def compute_likelihoods(self): 
        edidble_features_freq, poisonous_features_freq, total_edible_observations, total_poisonous_observations = self.compute_histograms()
        
        edidble_features_likelihoods = defaultdict(dict)
        poisonous_features_likelihoods = defaultdict(dict)
        
        for feature in edidble_features_freq:
            for value, freq in edidble_features_freq[feature].items():
                edidble_features_likelihoods[feature][value] = freq / total_edible_observations
        
        for feature in poisonous_features_freq:
            for value, freq in poisonous_features_freq[feature].items():
                poisonous_features_likelihoods[feature][value] = freq / total_poisonous_observations
                
        self.edidble_features_likelihoods, self.poisonous_features_likelihoods = edidble_features_likelihoods, poisonous_features_likelihoods
                    
        return self.edidble_features_likelihoods, self.poisonous_features_likelihoods
    
    def fit(self):
        self.compute_initial_probs()
        return self.compute_likelihoods()
    
    def predict(self, row: pd.Series):
        edidble_pred_score = np.log(self.edidble_prob)
        poisonous_pred_score = np.log(self.poisonous_prob)
        
        edidble_features_likelihoods = self.edidble_features_likelihoods
        poisonous_features_likelihoods = self.poisonous_features_likelihoods
        
        for feature, value in row.items():
            if feature == 'class': continue
            edidble_pred_score += np.log(edidble_features_likelihoods[feature].get(value, 1e-10))
            poisonous_pred_score += np.log(poisonous_features_likelihoods[feature].get(value, 1e-10))
            

        if edidble_pred_score > poisonous_pred_score:
            return 'e'
        else:
            return 'p'
            
            
    def test(self, test_data: pd.DataFrame) -> list:
        labels = []
        preds = []
        for _, row in test_data.iterrows():
            label = row['class']
            pred = self.predict(row=row)
            
            labels.append(label)
            preds.append(pred)
        return labels, preds
            
    def evaluate(self, test_results: list):
        labels = test_results[0]
        preds = test_results[1]

        print('===== accuracy =====')
        print(accuracy_score(labels, preds))
        print('====================\n')
        
        print('===== classification report =====')
        print(classification_report(labels, preds))
        print('=================================\n')
        
        print('===== confusion matrix =====')
        print(confusion_matrix(labels, preds))
        print('============================\n')    

In [ ]:
df = pd.read_csv('data/mushrooms.csv')
train_df, test_df = train_test_split(df, test_size=0.2)
nb = NaiveBayesClassifier(data=train_df)
nb.fit()
test_results = nb.test(test_data=test_df)
nb.evaluate(test_results)

In [ ]:
class NaiveBayesGaussian:
    def __init__(self):
        self.stats = {}
        self.priors = {}
        self.classes = None

    def fit(self, X, y):
        self.classes = np.unique(y)
        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = X_c.shape[0] / X.shape[0]
            self.stats[c] = {
                'mean': np.mean(X_c, axis=0),
                'var': np.var(X_c, axis=0) + 1e-4 
            }

    def predict(self, X):
        preds = [self._predict_one(x) for x in X]
        return np.array(preds)

    def _predict_one(self, x):
        posteriors = []
        for c in self.classes:
            prior = np.log(self.priors[c])
            mean = self.stats[c]['mean']
            var = self.stats[c]['var']
            likelihood = -0.5 * np.sum(np.log(2 * np.pi * var))
            likelihood -= 0.5 * np.sum(((x - mean)**2) / var)
            posteriors.append(prior + likelihood)
        return self.classes[np.argmax(posteriors)]

In [ ]:
encoded_df = encode_features(df)
pca = PCA(encoded_df)
pca_comps: np.ndarray = pca.compute_fprime()
y = df['class'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(pca_comps, y, test_size=0.2)
nb_all = NaiveBayesGaussian()
nb_all.fit(X_train, y_train)
print(f"Accuracy: {accuracy_score(y_test, nb_all.predict(X_test)):.4f}")



In [ ]:
fs = FeatureSelection()
X_fs = fs.select_top_variance(encoded_df.values, k=100)
X_train, X_test, y_train, y_test = train_test_split(X_fs, y, test_size=0.2)
nb_fs = NaiveBayesGaussian()
nb_fs.fit(X_train, y_train)
print(f"Accuracy: {accuracy_score(y_test, nb_fs.predict(X_test)):.4f}")